In [ ]:
from pathlib import Path
import yaml
import numpy as np

from mdx2.io import loadobj
import matplotlib.pyplot as plt

from mdx2.report.visualization import calc_isoavg, extract_central_slice, unique_slices

# load the site config
with open("site_config.yaml") as f:
    config = yaml.safe_load(f)

# project directory
project_directory = Path(config['project_directory'])

def name2key(dset_name):
    return dset_name.replace(' ','_').replace('-','').lower()

dataset_names = [
    'CHESS 2025-11-05',
    'CHESS 2025-09-24',
    'ALS 2025-10-15',
    'ALS 2025-11-25',
]

In [ ]:
# calculate isotropic part

def subtract_isotropic_part(da, s_iso, intensity_iso):
    s = np.sqrt(da.sx**2 + da.sy**2 + da.sz**2)
    intensity_interp = np.interp(s, s_iso, intensity_iso)
    da_subtracted = da.copy()
    da_subtracted.data = da.data - intensity_interp
    return da_subtracted

def ingest_merged_data(dset_root):

    crystal = loadobj(project_directory / dset_root / 'mac1_merge/mdx2/merged.nxs', 'crystal')
    symmetry =  loadobj(project_directory / dset_root / 'mac1_merge/mdx2/merged.nxs', 'symmetry')
    hkl_table =  loadobj(project_directory / dset_root / 'mac1_merge/mdx2/merged.nxs', 'hkl_table')

    s_iso, intensity_iso = calc_isoavg(hkl_table, crystal)

    slices = []
    for slice_index, coord in unique_slices(symmetry):
        intensity = extract_central_slice(
            hkl_table,
            symmetry,
            crystal,
            slice_index=slice_index,
            slice_coordinate=coord, # for some grids, 0.5 may round to 1/3, etc.
            signal="intensity",
        )
        intensity_subtracted = subtract_isotropic_part(intensity, s_iso, intensity_iso)
        slices.append({'intensity':intensity,'intensity_subtracted':intensity_subtracted})
    
    return {'isotropic':{'s':s_iso,'intensity':intensity_iso},'slices':slices}

data = {k:ingest_merged_data(name2key(k)) for k in dataset_names}


In [ ]:
plt.figure(figsize=(5.5,3))
for k, vals in data.items():
    normval = np.max(vals['isotropic']['intensity'])
    plt.plot(vals['isotropic']['s'], vals['isotropic']['intensity'] / normval, label=k)
plt.xlabel("1/d (Å$^{-1}$)")
plt.ylabel("Intensity (normalized)")
plt.legend()
plt.xlim(0,1)
plt.title('Isotropic component')

In [ ]:
# plot non-halo slices

for slice_index in [1,3]:
    for k, vals in data.items():
        fig, ax = plt.subplots(1,2, figsize=(10,4))
        normval = np.max(vals['isotropic']['intensity'])
        slice_data = vals['slices'][slice_index]
        s = np.sqrt(slice_data['intensity'].sx**2 + slice_data['intensity'].sy**2 + slice_data['intensity'].sz**2)
        (slice_data['intensity'].where(s<0.8,np.nan)/normval).plot.pcolormesh("sx", "sy", vmin=0, vmax=1.1, ax=ax[0])
        (slice_data['intensity_subtracted'].where(s<0.8,np.nan)/normval).plot.pcolormesh("sx", "sy", vmin=0.1, vmax=-0.1, cmap = 'RdBu_r', ax=ax[1])
        for a in ax:
            a.set_xlim(-0.8, 0.8)
            a.set_ylim(-0.8, 0.8)
            a.set_aspect('equal', adjustable='box')
        plt.suptitle(k)
        plt.show()

In [ ]:
# difference maps

# chess 2025-09-24 vs. als 2025-10-15

def crop2smax(da,smax):
    s = np.sqrt(da.sx**2 + da.sy**2 + da.sz**2)
    return da.where(s<smax,np.nan)

dataset_pairs = [
    (dataset_names[i], dataset_names[j])
    for i in range(len(dataset_names))
    for j in range(i + 1, len(dataset_names)) 
]

for dset1_name, dset2_name in dataset_pairs:
    fig, axes = plt.subplots(2,3, figsize=(14,8), sharex=True, sharey=True)
    for ax, slice_index in zip(axes,[1,3]):
    
        dset1_sub = data[dset1_name]['slices'][slice_index]['intensity_subtracted'].copy()
        dset1_sub /= data[dset1_name]['isotropic']['intensity'].max()
        
        dset2_sub = data[dset2_name]['slices'][slice_index]['intensity_subtracted'].copy()
        dset2_sub /= data[dset2_name]['isotropic']['intensity'].max()
        
        ds1_minus_ds2 = dset1_sub - dset2_sub
        try:
            # HACK ALERT! Why does it throw an error one way, but not the other?
            # TODO: determine which one is correct before trying it, or at least catch
            # the specific xarray error
            ds1_minus_ds2 = ds1_minus_ds2.assign_coords(dset1_sub.coords)
        except:
            ds1_minus_ds2 = ds1_minus_ds2.assign_coords(dset2_sub.coords)
        
        crop2smax(dset1_sub,0.8).plot.pcolormesh("sx", "sy", vmin=0.1, vmax=-0.1, cmap = 'RdBu_r',ax=ax[0])
        crop2smax(dset2_sub,0.8).plot.pcolormesh("sx", "sy", vmin=0.1, vmax=-0.1, cmap = 'RdBu_r',ax=ax[1])
        crop2smax(ds1_minus_ds2,0.8).plot.pcolormesh("sx", "sy", vmin=0.1, vmax=-0.1, cmap = 'RdBu_r',ax=ax[2])

    for axrows in axes:
        for ax in axrows:
            ax.set_aspect('equal', adjustable='box')
    axes[0][0].set_title(f"#1: {dset1_name}\n{axes[0][0].get_title()}")
    axes[0][1].set_title(f"#2: {dset2_name}\n{axes[0][1].get_title()}")
    axes[0][2].set_title(f"#1 - #2\n{axes[0][2].get_title()}")
    axes[0][0].set_xlim(-0.8,0.8)
    axes[0][0].set_ylim(-0.8,0.8)
    plt.tight_layout()
    plt.show()